# Train the "Orac" wake word (openWakeWord)

**Before running:** Runtime → Change runtime type → **T4 GPU**.
Then Runtime → **Run all** and come back in ~45–75 minutes.

Output: `orac_wake_models.zip` (auto-downloads at the end) containing
`orac.onnx`, `melspectrogram.onnx`, `embedding_model.onnx` — drop all
three into `public/models/` in the social-media-app repo.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install dependencies (ordered + guarded for current Colab images)
# Record Colab's preinstalled numpy so we can restore it after installs —
# stray pins from old audio packages can downgrade it and break the
# compiled stack (scipy/torch) with "numpy.dtype size changed" errors.
import numpy as np
open("/content/numpy_version.txt", "w").write(np.__version__)

!git clone https://github.com/dscripka/openWakeWord /content/openWakeWord
%cd /content/openWakeWord
!pip install -e .
%cd /content
!git clone https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# Training extras. Deliberately NOT pinned to 2023 versions and NOT
# including deep-phonemizer (numpy-breaking, unused by our pipeline).
!pip install mutagen torchinfo torchmetrics speechbrain==0.5.14 audiomentations acoustics onnx onnx2tf datasets pronouncing webrtcvad
!pip install -U torch-audiomentations
# piper-phonemize has no official Python 3.12 wheel; try community build too
!pip install piper-phonemize || pip install piper-phonemize-cross || echo "WARN: piper-phonemize unavailable — preflight will tell us"

# Restore Colab's exact numpy (no-deps so nothing else moves)
!pip install --force-reinstall --no-deps numpy==$(cat /content/numpy_version.txt)

# torchaudio 2.x shim: older audio libs call removed set_audio_backend()
import glob
shim = '''
try:
    import torchaudio
    if not hasattr(torchaudio, 'set_audio_backend'):
        torchaudio.set_audio_backend = lambda *a, **k: None
    if not hasattr(torchaudio, 'get_audio_backend'):
        torchaudio.get_audio_backend = lambda: 'soundfile'
except Exception:
    pass
'''
sp = glob.glob("/usr/local/lib/python3*/dist-packages")[0]
with open(f"{sp}/sitecustomize.py", "a") as f:
    f.write("\n" + shim)
print("installed")

In [ ]:
# 3. Training config — "Orac" + "Hey Orac". ABSOLUTE paths throughout:
# train.py runs from /content/openWakeWord, so relative paths break.
config = '''
target_phrase:
  - "orac"
  - "hey orac"
custom_negative_phrases:
  - "oracle"
  - "a rack"
  - "or act"
  - "okay"
model_name: "orac"
n_samples: 6000
n_samples_val: 1000
steps: 30000
max_negative_weight: 1500
target_accuracy: 0.7
target_recall: 0.5
target_false_positives_per_hour: 0.2
background_paths:
  - /content/audioset_16k
  - /content/fma
false_positive_validation_data_path: "/content/validation_set_features.npy"
feature_data_files:
  "ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50
piper_sample_generator_path: "/content/piper-sample-generator"
output_dir: "/content/orac_model"
tts_batch_size: 50
augmentation_batch_size: 16
augmentation_rounds: 1
layer_size: 32
'''
with open("/content/orac.yaml", "w") as f:
    f.write(config)
print("config written")

In [ ]:
# 4. Download negative/validation feature sets + create background audio
%cd /content
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Background noise via the stdlib `wave` module — deliberately NO numpy/scipy
# (immune to any binary-compat state) and no external downloads (URLs rot).
import wave, struct, random, os
random.seed(42)
for d in ["/content/fma", "/content/audioset_16k"]:
    os.makedirs(d, exist_ok=True)
    if not os.listdir(d):
        for i in range(60):
            with wave.open(f"{d}/noise_{i}.wav", "w") as w:
                w.setnchannels(1)
                w.setsampwidth(2)
                w.setframerate(16000)
                amp = random.uniform(300, 2500)
                prev = 0.0
                frames = bytearray()
                for _ in range(16000 * 10):  # 10s, low-pass shaped noise
                    prev = 0.85 * prev + random.uniform(-1, 1)
                    v = int(prev * amp)
                    frames += struct.pack("<h", max(-32768, min(32767, v)))
                w.writeframes(bytes(frames))
print("datasets ready")

In [ ]:
# 4.5 PREFLIGHT — verify every import and file the training needs.
# Fails fast with a named problem instead of a confusing mid-train error.
import importlib, sys, os
sys.path.append("/content/piper-sample-generator")
problems = []
for m in ["pronouncing", "webrtcvad", "torch_audiomentations", "speechbrain",
          "datasets", "mutagen", "acoustics", "audiomentations", "onnx",
          "torchmetrics", "generate_samples"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        problems.append(f"import {m}: {type(e).__name__}: {e}")
for f in ["/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
          "/content/validation_set_features.npy",
          "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",
          "/content/orac.yaml"]:
    if not os.path.exists(f):
        problems.append(f"missing file: {f}")
if not os.listdir("/content/fma"):
    problems.append("no background clips in /content/fma")
import numpy
print("numpy:", numpy.__version__)
import scipy  # binary-compat canary
print("scipy:", scipy.__version__, "(imports cleanly — no numpy mismatch)")
assert not problems, "PREFLIGHT FAILED — fix these before training:\n" + "\n".join(problems)
print("\nPREFLIGHT OK — run the training cell")

In [ ]:
# 5. Generate synthetic samples, augment, train (the long cell — ~30-60 min)
%cd /content/openWakeWord
!python openwakeword/train.py --training_config /content/orac.yaml --generate_clips
!python openwakeword/train.py --training_config /content/orac.yaml --augment_clips
!python openwakeword/train.py --training_config /content/orac.yaml --train_model
%cd /content

In [ ]:
# 6. Bundle the three runtime models and download
import shutil, glob, os
import openwakeword
%cd /content
os.makedirs("bundle", exist_ok=True)
# the trained classifier
trained = glob.glob("/content/orac_model/**/orac.onnx", recursive=True) + glob.glob("/content/**/orac.onnx", recursive=True)
assert trained, "orac.onnx not found — check the training cell output above"
shutil.copy(trained[0], "bundle/orac.onnx")
# the shared preprocessing models from the installed package (download if needed)
try:
    from openwakeword import utils as oww_utils
    if hasattr(oww_utils, "download_models"):
        oww_utils.download_models(["melspectrogram", "embedding_model"])
except Exception as e:
    print("download_models skipped:", e)
pkg = os.path.dirname(openwakeword.__file__)
for name in ["melspectrogram.onnx", "embedding_model.onnx"]:
    found = glob.glob(os.path.join(pkg, "resources", "models", name)) or glob.glob(f"/content/**/{name}", recursive=True)
    assert found, f"{name} not found"
    shutil.copy(found[0], f"bundle/{name}")
shutil.make_archive("orac_wake_models", "zip", "bundle")
print("Bundle contents:", os.listdir("bundle"))
from google.colab import files
files.download("/content/orac_wake_models.zip")

## Done

Unzip `orac_wake_models.zip` and put all three `.onnx` files into
`public/models/` in the social-media-app repo, then commit + push.
The app switches engines automatically.

If any cell failed, see `scripts/wake-training/README.md` — the upstream
notebook is the source of truth and this config pastes straight into it.